In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# ---------- Configuration ----------
WARD_BASE_URL = "https://integrity.ng/index.php/wards/browse"
POLLING_UNITS_BASE_URL = "https://integrity.ng/index.php/units/browse"
OUTPUT_FILE = "LGA_Wards_PollingUnits.xlsx"

# ---------- Scraping Functions ----------
def fetch_wards(page=1):
    url = f"{WARD_BASE_URL}/{page}"
    response = requests.get(url)
    if not response.ok:
        return []
    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find("table")
    if not table:
        return []
    rows = table.find_all("tr")[1:]  # Skip header
    wards = []
    for row in rows:
        cols = [td.get_text(strip=True) for td in row.find_all("td")]
        if len(cols) >= 4:
            wards.append({
                "LGA": cols[2],
                "Ward": cols[3]
            })
    return wards

def fetch_polling_units(page=1):
    url = f"{POLLING_UNITS_BASE_URL}/{page}"
    response = requests.get(url)
    if not response.ok:
        return []
    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find("table")
    if not table:
        return []
    rows = table.find_all("tr")[1:]  # Skip header
    pus = []
    for row in rows:
        cols = [td.get_text(strip=True) for td in row.find_all("td")]
        if len(cols) >= 5:
            pus.append({
                "LGA": cols[2],
                "Ward": cols[3],
                "Polling Station": cols[4]
            })
    return pus

# ---------- Main Logic ----------
def main():
    print("Fetching Wards...")
    all_wards = []
    for page in range(1, 100):  # Adjust max pages as needed
        wards = fetch_wards(page)
        if not wards:
            break
        all_wards.extend(wards)
        time.sleep(0.5)
    print(f"Total Wards Fetched: {len(all_wards)}")

    print("Fetching Polling Units...")
    all_pus = []
    for page in range(1, 200):  # Adjust max pages as needed
        pus = fetch_polling_units(page)
        if not pus:
            break
        all_pus.extend(pus)
        time.sleep(0.5)
    print(f"Total Polling Units Fetched: {len(all_pus)}")

    # Prepare LGAs sheet
    lga_list = sorted(set(entry["LGA"] for entry in all_wards))
    df_lga = pd.DataFrame({"LGA": lga_list})

    # Prepare Wards sheet
    df_wards = pd.DataFrame(all_wards).drop_duplicates()

    # Prepare Polling Units sheet
    df_pus = pd.DataFrame(all_pus).drop_duplicates()

    print("Saving to Excel...")
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        df_lga.to_excel(writer, sheet_name="LGAs", index=False)
        df_wards.to_excel(writer, sheet_name="Wards", index=False)
        df_pus.to_excel(writer, sheet_name="PollingUnits", index=False)

    print(f"✅ Done! File saved as: {OUTPUT_FILE}")

# ---------- Run ----------
if __name__ == "__main__":
    main()


In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ------------------------------------------
# SETTINGS
STATE_LGAS = {
    "Katsina": [
        "Bakori", "Batagarawa", "Batsari", "Baure", "Bindawa", "Charanchi",
        "Dan Musa", "Dandume", "Danja", "Daura", "Dutsi", "Dutsin-Ma",
        "Faskari", "Funtua", "Ingawa", "Jibia", "Kafur", "Kaita", "Kankara",
        "Kankia", "Katsina", "Kurfi", "Kusada", "Mai'Adua", "Malumfashi",
        "Mani", "Mashi", "Matazu", "Musawa", "Rimi", "Sabuwa", "Safana",
        "Sandamu", "Zango"
    ],
    # Add more states here if needed
}

WARD_BROWSE_URL = "https://integrity.ng/index.php/wards/browse"
INEC_PU_URL = "https://www.inecnigeria.org/polling-units/"
MAX_PAGES = 200
# ------------------------------------------

def init_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(options=options)

def fetch_wards(driver, state_name, lga_list, max_pages=MAX_PAGES):
    wards = []
    page = 0
    seen_rows = set()

    while page < max_pages:
        driver.get(f"{WARD_BROWSE_URL}?page={page}")
        try:
            table = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "table"))
            )
        except:
            print(f"❌ No table found on page {page}, stopping.")
            break

        rows = table.find_elements(By.TAG_NAME, "tr")[1:]
        if not rows:
            print(f"✅ No more rows on page {page}, stopping.")
            break

        page_has_new_data = False
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 4:
                lga = cols[2].text.strip()
                ward = cols[3].text.strip()
                if lga in lga_list:
                    row_id = f"{lga}-{ward}"
                    if row_id not in seen_rows:
                        wards.append({"LGA": lga, "Ward": ward})
                        seen_rows.add(row_id)
                        page_has_new_data = True

        print(f"✅ Page {page}: {len(rows)} rows, {len(wards)} total wards so far")
        if not page_has_new_data:
            print(f"✅ No new wards found on page {page}, stopping.")
            break

        page += 1
        time.sleep(1)

    return wards

def fetch_polling_units(driver, state_name, wards, lga_list):
    pus = []

    driver.get(INEC_PU_URL)
    try:
        state_dropdown = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.ID, "selState"))
        )
        driver.execute_script("arguments[0].scrollIntoView(true);", state_dropdown)
    except:
        print(f"❌ Could not load polling unit page for {state_name}. Skipping.")
        return pus

    try:
        state_select = Select(driver.find_element(By.ID, "selState"))
        state_select.select_by_visible_text(state_name)
        time.sleep(2)
    except:
        print(f"❌ State '{state_name}' not found in dropdown.")
        return pus

    for ward_data in wards:
        lga = ward_data['LGA']
        ward = ward_data['Ward']

        if lga not in lga_list:
            continue

        try:
            lga_select = Select(driver.find_element(By.ID, "selLga"))
            lga_select.select_by_visible_text(lga)
            time.sleep(1.5)

            ward_select = Select(driver.find_element(By.ID, "selWard"))
            ward_select.select_by_visible_text(ward)
            time.sleep(1)

            driver.find_element(By.ID, "btnSearch").click()
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "tblPu")))

            table = driver.find_element(By.ID, "tblPu")
            rows = table.find_elements(By.TAG_NAME, "tr")[1:]

            for r in rows:
                cols = r.find_elements(By.TAG_NAME, "td")
                if len(cols) >= 1:
                    pus.append({
                        "LGA": lga,
                        "Ward": ward,
                        "Polling Station": cols[0].text.strip()
                    })

            print(f"   ✔️ {lga} / {ward}: {len(rows)} polling units")
        except:
            print(f"   ⚠️ Skipped: {lga} / {ward}")
        time.sleep(1.5)

    # Final strict filter
    pus = [pu for pu in pus if pu["LGA"] in lga_list and any(w['Ward'] == pu['Ward'] for w in wards)]
    return pus

def save_to_excel(state_name, wards, pus):
    df_wards = pd.DataFrame(wards).drop_duplicates()
    df_pus = pd.DataFrame(pus).drop_duplicates()

    if "LGA" in df_wards.columns:
        df_lga = pd.DataFrame(sorted(df_wards["LGA"].unique()), columns=["LGA"])
    else:
        df_lga = pd.DataFrame(columns=["LGA"])
        print("⚠️ No LGAs found in ward data.")

    filename = f"{state_name}_LGA_Wards_PollingUnits.xlsx"
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        df_lga.to_excel(writer, sheet_name="LGAs", index=False)
        df_wards.to_excel(writer, sheet_name="Wards", index=False)
        df_pus.to_excel(writer, sheet_name="PollingUnits", index=False)

    print(f"\n✅ Saved to {filename}\n")

def run_for_state(state_name, lga_list):
    print(f"\n🚀 Starting scrape for: {state_name}")
    driver = init_driver()
    wards = fetch_wards(driver, state_name, lga_list)

    if not wards:
        print("⚠️ No wards found. Aborting.")
        driver.quit()
        return

    print(f"📌 Total Wards Collected: {len(wards)}")

    pus = fetch_polling_units(driver, state_name, wards, lga_list)
    print(f"📌 Total Polling Units Collected: {len(pus)}")

    save_to_excel(state_name, wards, pus)
    driver.quit()

# ----------------------------
# PICK A STATE INTERACTIVELY
# ----------------------------
user_state = input("Enter the state to scrape (e.g., Katsina): ").strip()

if user_state in STATE_LGAS:
    run_for_state(user_state, STATE_LGAS[user_state])
else:
    print(f"❌ State '{user_state}' not found in predefined list.")
    print("Available states:")
    for state in STATE_LGAS:
        print(" -", state)


In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ------------------------------------------
# SETTINGS: Update this list for more states
STATE_LGAS = {
    "Katsina": [
        "Bakori", "Batagarawa", "Batsari", "Baure", "Bindawa", "Charanchi",
        "Dan Musa", "Dandume", "Danja", "Daura", "Dutsi", "Dutsin-Ma",
        "Faskari", "Funtua", "Ingawa", "Jibia", "Kafur", "Kaita", "Kankara",
        "Kankia", "Katsina", "Kurfi", "Kusada", "Maiadua", "Malumfashi",
        "Mani", "Mashi", "Matazu", "Musawa", "Rimi", "Sabuwa", "Safana",
        "Sandamu", "Zango"
    ],
    # Add more states and LGAs here if needed
}
# ------------------------------------------

WARD_BROWSE_URL = "https://integrity.ng/index.php/wards/browse"
INEC_PU_URL = "https://www.inecnigeria.org/polling-units/"
MAX_PAGES = 200  # page scraping limit

# ------------------------------------------
def init_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(options=options)

def fetch_wards(driver, state_name, lga_list, max_pages=MAX_PAGES):
    wards = []
    page = 0
    seen_rows = set()
    normalized_lga_list = [l.strip().lower() for l in lga_list]

    while page < max_pages:
        driver.get(f"{WARD_BROWSE_URL}?page={page}")
        try:
            table = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "table"))
            )
        except:
            print(f"❌ No table found on page {page}, stopping.")
            break

        rows = table.find_elements(By.TAG_NAME, "tr")[1:]
        if not rows:
            print(f"✅ No more rows on page {page}, stopping.")
            break

        page_has_new_data = False
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) >= 4:
                lga = cols[2].text.strip()
                ward = cols[3].text.strip()

                if lga.strip().lower() in normalized_lga_list:
                    row_id = f"{lga}-{ward}"
                    if row_id not in seen_rows:
                        wards.append({"LGA": lga, "Ward": ward})
                        seen_rows.add(row_id)
                        page_has_new_data = True

        print(f"✅ Page {page}: {len(rows)} rows, {len(wards)} total wards so far")
        if not page_has_new_data:
            print(f"✅ No new wards found on page {page}, stopping.")
            break

        page += 1
        time.sleep(1)

    return wards

def fetch_polling_units(driver, state_name, wards, lga_list):
    pus = []

    driver.get(INEC_PU_URL)
    try:
        state_dropdown = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.ID, "selState"))
        )
        driver.execute_script("arguments[0].scrollIntoView(true);", state_dropdown)
    except:
        print(f"❌ Could not load polling unit page for {state_name}. Skipping.")
        return pus

    try:
        state_select = Select(driver.find_element(By.ID, "selState"))
        state_select.select_by_visible_text(state_name)
        time.sleep(2)
    except:
        print(f"❌ State '{state_name}' not found in dropdown.")
        return pus

    for ward_data in wards:
        lga = ward_data['LGA']
        ward = ward_data['Ward']

        if lga not in lga_list:
            continue

        try:
            lga_select = Select(driver.find_element(By.ID, "selLga"))
            lga_select.select_by_visible_text(lga)
            time.sleep(1.5)

            ward_select = Select(driver.find_element(By.ID, "selWard"))
            ward_select.select_by_visible_text(ward)
            time.sleep(1)

            driver.find_element(By.ID, "btnSearch").click()
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "tblPu")))

            table = driver.find_element(By.ID, "tblPu")
            rows = table.find_elements(By.TAG_NAME, "tr")[1:]

            for r in rows:
                cols = r.find_elements(By.TAG_NAME, "td")
                if len(cols) >= 1:
                    pus.append({
                        "LGA": lga,
                        "Ward": ward,
                        "Polling Station": cols[0].text.strip()
                    })

            print(f"   ✔️ {lga} / {ward}: {len(rows)} polling units")
        except:
            print(f"   ⚠️ Skipped: {lga} / {ward}")
        time.sleep(1.5)

    # Final cleanup filter
    pus = [pu for pu in pus if pu["LGA"] in lga_list and any(w['Ward'] == pu['Ward'] for w in wards)]
    return pus

def save_to_excel(state_name, wards, pus):
    df_wards = pd.DataFrame(wards).drop_duplicates()
    df_pus = pd.DataFrame(pus).drop_duplicates()

    if "LGA" in df_wards.columns:
        df_lga = pd.DataFrame(sorted(df_wards["LGA"].unique()), columns=["LGA"])
    else:
        df_lga = pd.DataFrame(columns=["LGA"])
        print("⚠️ No LGAs found in ward data.")

    filename = f"{state_name}_LGA_Wards_PollingUnits.xlsx"
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        df_lga.to_excel(writer, sheet_name="LGAs", index=False)
        df_wards.to_excel(writer, sheet_name="Wards", index=False)
        df_pus.to_excel(writer, sheet_name="PollingUnits", index=False)

    print(f"\n✅ Saved to {filename}\n")

def run_for_state(state_name, lga_list):
    print(f"\n🚀 Starting scrape for: {state_name}")
    driver = init_driver()
    wards = fetch_wards(driver, state_name, lga_list)

    if not wards:
        print("⚠️ No wards found. Aborting.")
        driver.quit()
        return

    print(f"📌 Total Wards Collected: {len(wards)}")

    pus = fetch_polling_units(driver, state_name, wards, lga_list)
    print(f"📌 Total Polling Units Collected: {len(pus)}")

    save_to_excel(state_name, wards, pus)
    driver.quit()

# ------------------------------------------
# INTERACTIVE USER INPUT
# ------------------------------------------
user_state = input("Enter the state to scrape (e.g., Katsina): ").strip()

if user_state in STATE_LGAS:
    run_for_state(user_state, STATE_LGAS[user_state])
else:
    print(f"❌ State '{user_state}' not found in predefined list.")
    print("Available states:")
    for state in STATE_LGAS:
        print(" -", state)


In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

WARD_URL = "https://integrity.ng/index.php/wards/browse"
PU_URL = "https://integrity.ng/index.php/units/browse"
MAX_PAGES = 200


def init_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def get_all_states():
    print("📌 Fetching all states from integrity.ng/wards")
    driver = init_driver()
    states = set()
    page = 0

    while page < MAX_PAGES:
        driver.get(f"{WARD_URL}?page={page}")
        try:
            table = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "table")))
            rows = table.find_elements(By.TAG_NAME, "tr")[1:]
            if not rows:
                break

            for row in rows:
                cols = row.find_elements(By.TAG_NAME, "td")
                if len(cols) >= 4:
                    state = cols[1].text.strip()
                    if state:
                        states.add(state)

            page += 1
            time.sleep(1)

        except:
            break

    driver.quit()
    return sorted(list(states))


def fetch_wards(driver, state_name):
    wards = []
    page = 0
    seen = set()

    while page < MAX_PAGES:
        driver.get(f"{WARD_URL}?page={page}")
        try:
            table = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "table")))
            rows = table.find_elements(By.TAG_NAME, "tr")[1:]
            if not rows:
                break

            for row in rows:
                cols = row.find_elements(By.TAG_NAME, "td")
                if len(cols) >= 4:
                    state = cols[1].text.strip()
                    lga = cols[2].text.strip()
                    ward = cols[3].text.strip()

                    if state.lower() == state_name.lower():
                        key = f"{lga}-{ward}"
                        if key not in seen:
                            wards.append({"LGA": lga, "Ward": ward})
                            seen.add(key)

            print(f"✅ Page {page}: {len(wards)} total wards so far")
            page += 1
            time.sleep(1)

        except:
            break

    return wards


def fetch_polling_units(driver, state_name):
    pus = []
    seen = set()
    page = 0

    while page < MAX_PAGES:
        driver.get(f"{PU_URL}?page={page}")
        try:
            table = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "table")))
            rows = table.find_elements(By.TAG_NAME, "tr")[1:]
            if not rows:
                break

            for row in rows:
                cols = row.find_elements(By.TAG_NAME, "td")
                if len(cols) >= 5:
                    state = cols[1].text.strip()
                    ward = cols[3].text.strip()
                    pu = cols[4].text.strip()

                    if state.lower() == state_name.lower():
                        key = f"{ward}-{pu}"
                        if key not in seen:
                            pus.append({"Ward": ward, "Polling Unit": pu})
                            seen.add(key)

            print(f"✅ PU Page {page}: {len(pus)} total polling units so far")
            page += 1
            time.sleep(1)

        except:
            break

    return pus


def extract_and_save_state_data(state_name):
    print(f"\n🚀 Starting scrape for: {state_name}")
    driver = init_driver()

    # Step 1: Extract Wards
    wards = fetch_wards(driver, state_name)
    if not wards:
        print(f"⚠️ No wards found for {state_name}. Skipping.")
        driver.quit()
        return

    print(f"📌 Collected {len(wards)} wards")
    lgas = sorted(set([w["LGA"] for w in wards]))

    # Step 2: Extract Polling Units
    pus = fetch_polling_units(driver, state_name)
    print(f"📌 Collected {len(pus)} polling units")

    driver.quit()

    # Step 3: Save Excel
    df_lga = pd.DataFrame(lgas, columns=["LGA"])
    df_wards = pd.DataFrame(wards).drop_duplicates()
    df_pus = pd.DataFrame(pus).drop_duplicates()

    file_name = f"{state_name}_LGAs_Wards_PUs.xlsx"
    with pd.ExcelWriter(file_name, engine="openpyxl") as writer:
        df_lga.to_excel(writer, sheet_name="LGAs", index=False)
        df_wards.to_excel(writer, sheet_name="LGA_Wards", index=False)
        df_pus.to_excel(writer, sheet_name="Wards_PUs", index=False)

    print(f"✅ Saved {file_name} ({len(df_lga)} LGAs, {len(df_wards)} wards, {len(df_pus)} polling units)")


if __name__ == "__main__":
    all_states = get_all_states()
    for state in all_states:
        try:
            extract_and_save_state_data(state)
        except Exception as e:
            print(f"❌ Failed for {state}: {e}")


In [ ]:
import pandas as pd
import os

# Path to your completed CSV (update this if needed)
INPUT_CSV = "polling_units_integrity.csv"
OUTPUT_DIR = "C:/Users/JARE WORKS/Documents/Jare Js journey/OUTPUT_STATESs"

# Create output folder
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load the full dataset
df = pd.read_csv(INPUT_CSV)

# Group by state and create separate Excel files
for state in df['State'].unique():
    df_state = df[df['State'] == state]

    # Sheet 1: Unique LGAs
    sheet1 = df_state[['LGA']].drop_duplicates().sort_values(by="LGA")

    # Sheet 2: Unique LGA + Ward pairs
    sheet2 = df_state[['LGA', 'Ward']].drop_duplicates().sort_values(by=["LGA", "Ward"])

    # Sheet 3: Full detail (LGA, Ward, Polling Unit)
    sheet3 = df_state[['LGA', 'Ward', 'Polling Station']].drop_duplicates().sort_values(by=["LGA", "Ward", "Polling Station"])

    # Optional: Clean filename
    state_clean = state.replace(" ", "_").replace("/", "_")

    # Create Excel writer
    output_path = os.path.join(OUTPUT_DIR, f"{state_clean}.xlsx")
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        sheet1.to_excel(writer, sheet_name="LGA", index=False)
        sheet2.to_excel(writer, sheet_name="LGA_Wards", index=False)
        sheet3.to_excel(writer, sheet_name="Polling_Units", index=False)

    print(f"✅ Saved: {output_path}")

print("\n✅ All states processed and saved into separate Excel files.")


In [ ]:
import pandas as pd
import os

# Load the main CSV
df = pd.read_csv("polling_units_integrity.csv")

# Create output folder
output_folder = "OUTPUT_STATES"
os.makedirs(output_folder, exist_ok=True)

# Group by State
for state, state_df in df.groupby("State"):
    # Convert state to string and clean it up
    state_str = str(state).strip().replace(" ", "_")

    # Skip invalid or empty state names
    if not state_str or state_str.lower() in ["nan", "none", ""]:
        print(f"⚠️ Skipping invalid state name: {state}")
        continue

    print(f"📂 Processing state: {state_str}")

    # Prepare data for each sheet
    sheet1 = state_df[["LGA"]].drop_duplicates().reset_index(drop=True)
    sheet2 = state_df[["LGA", "Ward"]].drop_duplicates().reset_index(drop=True)
    sheet3 = state_df[
        ["LGA", "Ward", "Polling Station"]
    ].reset_index(drop=True)

    # Define Excel file path
    file_path = os.path.join(output_folder, f"{state_str}.xlsx")

    # Write to Excel with 3 sheets
    with pd.ExcelWriter(file_path, engine="xlsxwriter") as writer:
        sheet1.to_excel(writer, index=False, sheet_name="LGA")
        sheet2.to_excel(writer, index=False, sheet_name="LGA_WARD")
        sheet3.to_excel(writer, index=False, sheet_name="LGA_WARD_PU")

    print(f"✅ Saved {file_path}")

print("\n🎉 All valid states processed successfully — one Excel per state with 3 sheets.")


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("polling_units_integrity.csv")
df

In [ ]:
df["Ward"]= df["Ward"].drop_duplicates()
df["Ward"]

In [ ]:
import pandas as pd
import re

# Read your CSV (no header, because your data looks raw)
df = pd.read_csv("polling_units_integrity.csv", header=None)

# Rename the first column
df.columns = ['raw_text']

# Define cleaner
def clean_repeated_words(text):
    if not isinstance(text, str):
        return text
    parts = re.split(r'[-,]', text)
    parts = [p.strip() for p in parts if p.strip()]
    cleaned = []
    for p in parts:
        if not cleaned or cleaned[-1].lower() != p.lower():
            cleaned.append(p)
    return ', '.join(cleaned)

# Apply function
df['cleaned'] = df['Polling Station'].apply(clean_repeated_words)

# Show results
print(df.head())



In [ ]:
df["cleaned"]

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("polling_units_integrity_all_pages.csv")

# Basic inspection
print("Shape of the dataset:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nSample rows:\n", df.head(10))
print("\nNumber of missing values per column:\n", df.isnull().sum())


Shape of the dataset: (119706, 7)

Column names: ['State', 'LGA', 'Ward', 'Polling Station', 'Code', 'Agent Phone', 'PVC']

Sample rows:
    State   LGA       Ward Polling Station  \
0      1  ABIA  ABA NORTH  ARIARIA MARKET   
1      2  ABIA  ABA NORTH  ARIARIA MARKET   
2      3  ABIA  ABA NORTH  ARIARIA MARKET   
3      4  ABIA  ABA NORTH  ARIARIA MARKET   
4      5  ABIA  ABA NORTH  ARIARIA MARKET   
5      6  ABIA  ABA NORTH  ARIARIA MARKET   
6      7  ABIA  ABA NORTH  ARIARIA MARKET   
7      8  ABIA  ABA NORTH  ARIARIA MARKET   
8      9  ABIA  ABA NORTH  ARIARIA MARKET   
9     10  ABIA  ABA NORTH  ARIARIA MARKET   

                                       Code  Agent Phone  PVC  
0                   B.T.C-SCHOOL PREMISES I            7  NaN  
1                  B.T.C-SCHOOL PREMISES II            8  NaN  
2                 B.T.C-SCHOOL PREMISES III            9  NaN  
3                  B.T.C-SCHOOL PREMISES IV           10  NaN  
4                   B.T.C-SCHOOL PREMISES V   

In [ ]:
# Remove perfect duplicates (across all columns)
df = df.drop_duplicates(keep='first')

# Optional: remove duplicates based on specific fields
df = df.drop_duplicates(subset=['Ward', 'Polling Station'], keep='first')

print("✅ After removing duplicates, new shape:", df.shape)


✅ After removing duplicates, new shape: (8810, 7)


In [ ]:
df["State"] =df["State"].astype(str)
df

,State,LGA,Ward,Polling Station,Code,Agent Phone,PVC
0,1,ABIA,ABA NORTH,ARIARIA MARKET,B.T.C-SCHOOL PREMISES I,7,NaN
48,49,ABIA,ABA NORTH,EZIAMA,ABIA POLY - ABIA POLY II,6,NaN
71,72,ABIA,ABA NORTH,INDUSTRIAL AREA,BATA PREMISES - BATA PREMISES I,5,NaN
100,101,ABIA,ABA NORTH,OGBOR I,HIS STEPS PRIM.SCHOOL - SCHOOL PREMISES I,11,NaN
127,128,ABIA,ABA NORTH,OGBOR II,FEDERAL HOUSING ESTATE-SEC.PREMISES I,22,NaN
...,...,...,...,...,...,...,...
119626,119627,ZAMFARA,ZURMI,KWASHBAWA,DAUDAWA / GARKAR LIMAN,9,NaN
119641,119642,ZAMFARA,ZURMI,MASHEM,GANBIRU / YAR KASUWA,9,NaN
119651,119652,ZAMFARA,ZURMI,RUKUDAWA,DUNMBURUM / TOWN PRIMARY SCHOOL,4,NaN
119666,119667,ZAMFARA,ZURMI,YAN BUKI/ DUTSI,BAYAN DUTSI / YAR YARA,16,NaN


In [ ]:
df = df.drop(["PVC","Agent Phone"], axis=1)

In [ ]:
df["Ward"] =df["Ward"].rename("state")
df


,Unnamed: 0,State,Lga,Ward,Polling Station
0,117191,State,ZAMFARA,ANKA,BAGEGA
1,117204,State,ZAMFARA,ANKA,BARAYAR-ZAKI
2,117221,State,ZAMFARA,ANKA,DAN GALADIMA
3,117228,State,ZAMFARA,ANKA,GALADIMA
4,117237,State,ZAMFARA,ANKA,MAGAJI
...,...,...,...,...,...
142,119626,State,ZAMFARA,ZURMI,KWASHBAWA
143,119641,State,ZAMFARA,ZURMI,MASHEM
144,119651,State,ZAMFARA,ZURMI,RUKUDAWA
145,119666,State,ZAMFARA,ZURMI,YAN BUKI/ DUTSI


In [ ]:
import pandas as pd
import os

# === 1. Load your dataset ===
file_path = "result in progress"
data = pd.read_csv(file_path)

# === 2. Define the column to group by ===
lga_col = "LGA"   # <-- changed from "State"

# === 3. Create output folder ===
output_dir = "OUTPUT_STATES"
os.makedirs(output_dir, exist_ok=True)

# === 4. Clean data ===
data[lga_col] = data[lga_col].astype(str).str.strip()

# === 5. Initialize ===
current_lga = None
lga_rows = []

# === 6. Helper function to validate LGA names ===
def valid_lga(value):
    """Check that LGA value is not numeric or empty."""
    if not isinstance(value, str):
        return False
    value = value.strip()
    if value == "" or value.isnumeric():
        return False
    return True

# === 7. Iterate through rows ===
for idx, row in data.iterrows():
    lga_value = str(row[lga_col]).strip()

    # Skip invalid or numeric LGA values
    if not valid_lga(lga_value):
        print(f"⚠️ Skipping invalid LGA at row {idx+1}: '{lga_value}'")
        continue

    # First valid LGA
    if current_lga is None:
        current_lga = lga_value

    # Same LGA as before → keep adding
    if lga_value.lower() == current_lga.lower():
        lga_rows.append(row)
    else:
        # LGA changed → save previous rows
        if lga_rows:
            df_lga = pd.DataFrame(lga_rows)
            output_path = os.path.join(output_dir, f"{current_lga}.csv")

            # Append if file already exists
            if os.path.exists(output_path):
                df_lga.to_csv(output_path, mode="a", header=False, index=False)
            else:
                df_lga.to_csv(output_path, index=False)

            print(f"✅ Saved {len(lga_rows)} rows for {current_lga}")

        # Reset for new LGA
        current_lga = lga_value
        lga_rows = [row]

# === 8. Save the last group ===
if lga_rows:
    df_lga = pd.DataFrame(lga_rows)
    output_path = os.path.join(output_dir, f"{current_lga}.csv")

    if os.path.exists(output_path):
        df_lga.to_csv(output_path, mode="a", header=False, index=False)
    else:
        df_lga.to_csv(output_path, index=False)

    print(f"✅ Saved {len(lga_rows)} rows for {current_lga}")

print("\n🎯 Done! All rows processed — grouped sequentially by LGA.")


✅ Saved 184 rows for ABIA
✅ Saved 226 rows for ADAMAWA
✅ Saved 329 rows for AKWA IBOM
✅ Saved 327 rows for ANAMBRA
✅ Saved 212 rows for BAUCHI
✅ Saved 105 rows for BAYELSA
✅ Saved 276 rows for BENUE
✅ Saved 312 rows for BORNO
✅ Saved 193 rows for CROSS RIVER
✅ Saved 270 rows for DELTA
✅ Saved 171 rows for EBONYI
✅ Saved 192 rows for EDO
✅ Saved 177 rows for EKITI
✅ Saved 260 rows for ENUGU
✅ Saved 62 rows for FCT
✅ Saved 114 rows for GOMBE
✅ Saved 305 rows for IMO
✅ Saved 287 rows for JIGAWA
✅ Saved 255 rows for KADUNA
✅ Saved 484 rows for KANO
✅ Saved 361 rows for KATSINA
✅ Saved 225 rows for KEBBI
✅ Saved 239 rows for KOGI
✅ Saved 193 rows for KWARA
✅ Saved 245 rows for LAGOS
✅ Saved 147 rows for NASARAWA
✅ Saved 274 rows for NIGER
✅ Saved 236 rows for OGUN
✅ Saved 203 rows for ONDO
✅ Saved 332 rows for OSUN
✅ Saved 351 rows for OYO
✅ Saved 207 rows for PLATEAU
✅ Saved 319 rows for RIVERS
✅ Saved 244 rows for SOKOTO
✅ Saved 168 rows for TARABA
✅ Saved 178 rows for YOBE
✅ Saved 147 ro

In [ ]:
import pandas as pd
import os

# === 1. Load your dataset ===
file_path = "result in progress"
data = pd.read_csv(file_path)

# === 2. Define the column to group by ===
lga_col = "LGA"

# === 3. Create output folder ===
output_dir = "OUTPUT_STATES"
os.makedirs(output_dir, exist_ok=True)

# === 4. Clean data ===
data[lga_col] = data[lga_col].astype(str).str.strip()

# === 5. Initialize ===
current_lga = None
lga_rows = []

# === 6. Helper function to validate LGA names ===
def valid_lga(value):
    """Check that LGA value is not numeric or empty."""
    if not isinstance(value, str):
        return False
    value = value.strip()
    if value == "" or value.isnumeric():
        return False
    return True

# === 7. Iterate through rows ===
for idx, row in data.iterrows():
    lga_value = str(row[lga_col]).strip()

    # Skip invalid or numeric LGA values
    if not valid_lga(lga_value):
        print(f"⚠️ Skipping invalid LGA at row {idx+1}: '{lga_value}'")
        continue

    # First valid LGA
    if current_lga is None:
        current_lga = lga_value

    # Same LGA as before → keep adding
    if lga_value.lower() == current_lga.lower():
        lga_rows.append(row.to_dict())  # <-- FIXED: store full row as dict
    else:
        # LGA changed → save previous rows
        if lga_rows:
            df_lga = pd.DataFrame(lga_rows)

            # Move 'code' column to the end if it exists
            if 'code' in df_lga.columns:
                cols = [col for col in df_lga.columns if col != 'code'] + ['code']
                df_lga = df_lga[cols]

            output_path = os.path.join(output_dir, f"{current_lga}.csv")

            if os.path.exists(output_path):
                df_lga.to_csv(output_path, mode="a", header=False, index=False)
            else:
                df_lga.to_csv(output_path, index=False)

            print(f"✅ Saved {len(lga_rows)} rows for {current_lga}")

        # Reset for new LGA
        current_lga = lga_value
        lga_rows = [row.to_dict()]  # <-- FIXED: store new row as dict

# === 8. Save the last group ===
if lga_rows:
    df_lga = pd.DataFrame(lga_rows)

    if 'code' in df_lga.columns:
        cols = [col for col in df_lga.columns if col != 'code'] + ['code']
        df_lga = df_lga[cols]

    output_path = os.path.join(output_dir, f"{current_lga}.csv")

    if os.path.exists(output_path):
        df_lga.to_csv(output_path, mode="a", header=False, index=False)
    else:
        df_lga.to_csv(output_path, index=False)

    print(f"✅ Saved {len(lga_rows)} rows for {current_lga}")

print("\n🎯 Done! All rows processed — grouped sequentially by LGA.")


✅ Saved 184 rows for ABIA
✅ Saved 226 rows for ADAMAWA
✅ Saved 329 rows for AKWA IBOM
✅ Saved 327 rows for ANAMBRA
✅ Saved 212 rows for BAUCHI
✅ Saved 105 rows for BAYELSA
✅ Saved 276 rows for BENUE
✅ Saved 312 rows for BORNO
✅ Saved 193 rows for CROSS RIVER
✅ Saved 270 rows for DELTA
✅ Saved 171 rows for EBONYI
✅ Saved 192 rows for EDO
✅ Saved 177 rows for EKITI
✅ Saved 260 rows for ENUGU
✅ Saved 62 rows for FCT
✅ Saved 114 rows for GOMBE
✅ Saved 305 rows for IMO
✅ Saved 287 rows for JIGAWA
✅ Saved 255 rows for KADUNA
✅ Saved 484 rows for KANO
✅ Saved 361 rows for KATSINA
✅ Saved 225 rows for KEBBI
✅ Saved 239 rows for KOGI
✅ Saved 193 rows for KWARA
✅ Saved 245 rows for LAGOS
✅ Saved 147 rows for NASARAWA
✅ Saved 274 rows for NIGER
✅ Saved 236 rows for OGUN
✅ Saved 203 rows for ONDO
✅ Saved 332 rows for OSUN
✅ Saved 351 rows for OYO
✅ Saved 207 rows for PLATEAU
✅ Saved 319 rows for RIVERS
✅ Saved 244 rows for SOKOTO
✅ Saved 168 rows for TARABA
✅ Saved 178 rows for YOBE
✅ Saved 147 ro

In [ ]:
import os
import pandas as pd

# === 1. Load original dataset with the full 'code' column ===
original_csv = "result in progress"
original_df = pd.read_csv(original_csv)
original_df["LGA"] = original_df["LGA"].astype(str).str.strip()

# === 2. Folder containing the split CSVs by LGA ===
split_folder = "OUTPUT_STATES"

# === 3. Loop through each file and append 'code' column ===
for filename in os.listdir(split_folder):
    if filename.endswith(".csv"):
        lga_name = filename.replace(".csv", "").strip()
        file_path = os.path.join(split_folder, filename)

        try:
            # Load the split CSV file
            split_df = pd.read_csv(file_path)

            # Get rows in the original data for the current LGA
            lga_match = original_df[original_df["LGA"].str.lower() == lga_name.lower()].copy()

            # Check that number of rows match
            if len(split_df) != len(lga_match):
                print(f"⚠️ Skipping {filename} — row count mismatch ({len(split_df)} vs {len(lga_match)})")
                continue

            # Copy the 'code' column from original dataset
            split_df["code"] = lga_match["code"].values

            # Save the updated file (overwrite)
            split_df.to_csv(file_path, index=False)
            print(f"✅ Code column added to {filename}")

        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")

print("\n🎯 Done! All matching files updated with the 'code' column.")


❌ Error processing ABIA.csv: 'code'
❌ Error processing ADAMAWA.csv: 'code'
❌ Error processing AKWA IBOM.csv: 'code'
❌ Error processing ANAMBRA.csv: 'code'
❌ Error processing BAUCHI.csv: 'code'
❌ Error processing BAYELSA.csv: 'code'
❌ Error processing BENUE.csv: 'code'
❌ Error processing BORNO.csv: 'code'
❌ Error processing CROSS RIVER.csv: 'code'
❌ Error processing DELTA.csv: 'code'
❌ Error processing EBONYI.csv: 'code'
❌ Error processing EDO.csv: 'code'
❌ Error processing EKITI.csv: 'code'
❌ Error processing ENUGU.csv: 'code'
❌ Error processing FCT.csv: 'code'
❌ Error processing GOMBE.csv: 'code'
❌ Error processing IMO.csv: 'code'
❌ Error processing JIGAWA.csv: 'code'
❌ Error processing KADUNA.csv: 'code'
❌ Error processing KANO.csv: 'code'
❌ Error processing KATSINA.csv: 'code'
❌ Error processing KEBBI.csv: 'code'
❌ Error processing KOGI.csv: 'code'
❌ Error processing KWARA.csv: 'code'
❌ Error processing LAGOS.csv: 'code'
❌ Error processing NASARAWA.csv: 'code'
❌ Error processing NIG

In [ ]:
import pandas as pd
import os

# === CONFIGURATION ===
input_folder = "OUTPUT_STATES"   # folder containing your CSV files
output_folder = "converted_excels"  # folder to save Excel files

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Loop through all CSV files in the folder
for file in os.listdir(input_folder):
    if file.lower().endswith(".csv"):
        csv_path = os.path.join(input_folder, file)
        print(f"Processing: {file}")

        # Read CSV
        df = pd.read_csv(csv_path)

        # Normalize column names (title case and strip spaces)
        df.columns = [col.strip().title() for col in df.columns]

        # Ensure required columns exist
        if not {"Lga", "Ward", "Polling Station"}.issubset(df.columns):
            print(f"❌ Skipping {file} (missing one of 'LGA', 'Ward', 'Polling Station')")
            continue

        # === Build sheets ===
        # Sheet 1: LGA (unique)
        lga_df = pd.DataFrame(sorted(df["Lga"].dropna().unique()), columns=["LGA"])

        # Sheet 2: WARDS (unique per LGA)
        wards_df = (
            df[["Lga", "Ward"]]
            .drop_duplicates()
            .sort_values(["Lga", "Ward"])
            .reset_index(drop=True)
        )

        # Sheet 3: PU (LGA, Ward, Polling Station)
        pu_df = (
            df[["Lga", "Ward", "Polling Station"]]
            .dropna()
            .rename(columns={"Polling Station": "PU"})
            .reset_index(drop=True)
        )

        # === Save to Excel ===
        output_path = os.path.join(output_folder, file.replace(".csv", "_katsina_format.xlsx"))
        with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
            lga_df.to_excel(writer, index=False, sheet_name="LGA")
            wards_df.to_excel(writer, index=False, sheet_name="WARDS")
            pu_df.to_excel(writer, index=False, sheet_name="PU")

        print(f"✅ Saved: {output_path}")

print("🎉 All CSVs converted successfully!")


🎉 All CSVs converted successfully!


In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

URL = "https://www.inecnigeria.org/polling-units/"
OUTPUT_FILE = "inec_polling_units_fixed.csv"
WAIT_TIME = 10

# Setup Selenium
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
# chrome_options.add_argument("--headless")  # optional
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
driver.get(URL)
wait = WebDriverWait(driver, WAIT_TIME)

data = []

try:
    # Wait for first dropdown (State)
    state_dropdown = wait.until(EC.presence_of_element_located((By.XPATH, "//select[contains(@name,'state')]")))
    state_select = Select(state_dropdown)
    print(f"Found {len(state_select.options)-1} states")

    # Limit to first state for test (you can change this later)
    for s_index in range(1, 2):  
        state_name = state_select.options[s_index].text.strip()
        print(f"\n🌍 State: {state_name}")
        state_select.select_by_index(s_index)
        time.sleep(2)

        # Wait for LGA dropdown
        lga_dropdown = wait.until(EC.presence_of_element_located((By.XPATH, "//select[contains(@name,'lga')]")))
        lga_select = Select(lga_dropdown)

        for l_index in range(1, 2):  # one LGA for test
            lga_name = lga_select.options[l_index].text.strip()
            print(f"  🏛 LGA: {lga_name}")
            lga_select.select_by_index(l_index)
            time.sleep(2)

            # Wait for Ward dropdown
            ward_dropdown = wait.until(EC.presence_of_element_located((By.XPATH, "//select[contains(@name,'ward')]")))
            ward_select = Select(ward_dropdown)

            for w_index in range(1, len(ward_select.options)):
                ward_name = ward_select.options[w_index].text.strip()
                print(f"    🏘 Ward: {ward_name}")

                # Select Ward
                ward_select.select_by_index(w_index)
                time.sleep(1)

                # Find and click the "Click to search" button
                search_button = driver.find_element(By.XPATH, "//button[contains(text(),'Click to search')]")
                driver.execute_script("arguments[0].scrollIntoView(true);", search_button)
                time.sleep(1)
                search_button.click()
                time.sleep(3)

                # Wait for polling unit table
                try:
                    table = wait.until(EC.presence_of_element_located((By.XPATH, "//table")))
                    rows = table.find_elements(By.TAG_NAME, "tr")

                    for row in rows[1:]:
                        cols = row.find_elements(By.TAG_NAME, "td")
                        if len(cols) >= 3:
                            pu_number = cols[0].text.strip()
                            pu_name = cols[1].text.strip()
                            status = cols[2].text.strip()
                            data.append([state_name, lga_name, ward_name, pu_number, pu_name, status])

                except Exception:
                    print("      ⚠ No table found for this ward.")
                time.sleep(1)

except Exception as e:
    print("❌ Error during scraping:", e)

# Save results
df = pd.DataFrame(data, columns=["State", "LGA", "Ward", "PU Number", "Polling Unit", "Status"])
df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Done. Saved {len(df)} polling units to {OUTPUT_FILE}")

driver.quit()


❌ Error during scraping: Message: 
Stacktrace:
	GetHandleVerifier [0x0xddfea3+66515]
	GetHandleVerifier [0x0xddfee4+66580]
	(No symbol) [0x0xbcdc48]
	(No symbol) [0x0xc18704]
	(No symbol) [0x0xc18aab]
	(No symbol) [0x0xc5f482]
	(No symbol) [0x0xc3b214]
	(No symbol) [0x0xc5cba7]
	(No symbol) [0x0xc3afc6]
	(No symbol) [0x0xc0c2ca]
	(No symbol) [0x0xc0d154]
	GetHandleVerifier [0x0x10373d3+2521347]
	GetHandleVerifier [0x0x1032353+2500739]
	GetHandleVerifier [0x0xe07cf4+229924]
	GetHandleVerifier [0x0xdf8258+165768]
	GetHandleVerifier [0x0xdfed0d+193085]
	GetHandleVerifier [0x0xde81b8+100072]
	GetHandleVerifier [0x0xde8350+100480]
	GetHandleVerifier [0x0xdd260a+11066]
	BaseThreadInitThunk [0x0x75e75d49+25]
	RtlInitializeExceptionChain [0x0x771ed6db+107]
	RtlGetAppContainerNamedObjectPath [0x0x771ed661+561]


✅ Done. Saved 0 polling units to inec_polling_units_fixed.csv


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

chrome_options = Options()
chrome_options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

driver.get("https://www.inecnigeria.org/polling-units/")
print(driver.page_source[:5000])  # print first 5000 characters of page HTML
driver.quit()


In [23]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


# ==============================
# CONFIGURATION
# ==============================
BASE_URL = "https://www.inecnigeria.org/polling-units/"
OUTPUT_FOLDER = "INEC_PollingUnits"
WAIT_TIME = 15


# ==============================
# SETUP WEBDRIVER
# ==============================
def create_driver():
    chrome_options = Options()
    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--disable-infobars")
    chrome_options.add_argument("--disable-extensions")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver


# ==============================
# SCRAPE A SINGLE STATE
# ==============================
def scrape_state(driver, state_name):
    wait = WebDriverWait(driver, WAIT_TIME)

    try:
        driver.get(BASE_URL)
        time.sleep(3)

        # --- Select the State ---
        state_dropdown = wait.until(EC.presence_of_element_located((By.ID, "statePoll")))
        Select(state_dropdown).select_by_visible_text(state_name)
        time.sleep(3)
        print(f"✅ Selected state: {state_name}")

        # --- Get LGA dropdown ---
        lga_dropdown = wait.until(EC.presence_of_element_located((By.ID, "lgaPoll")))
        lgas = [opt.text.strip() for opt in lga_dropdown.find_elements(By.TAG_NAME, "option") if "Choose" not in opt.text]
        print(f"Found {len(lgas)} LGAs in {state_name}")

        all_polling_units = []
        lga_list = []
        ward_list = []

        for lga in lgas:
            try:
                Select(lga_dropdown).select_by_visible_text(lga)
                time.sleep(2)

                # --- Wait for Wards ---
                ward_dropdown = wait.until(EC.presence_of_element_located((By.ID, "wardPoll")))
                wards = [opt.text.strip() for opt in ward_dropdown.find_elements(By.TAG_NAME, "option") if "Choose" not in opt.text]
                print(f"  ▪ LGA: {lga} → {len(wards)} wards found")

                # Save for Sheet1 and Sheet2
                lga_list.append({"State": state_name, "LGA": lga})
                for ward in wards:
                    ward_list.append({"State": state_name, "LGA": lga, "Ward": ward})

                    try:
                        Select(ward_dropdown).select_by_visible_text(ward)
                        time.sleep(2)

                        # Click Search
                        search_button = driver.find_element(By.ID, "search_poll_btn")
                        driver.execute_script("arguments[0].click();", search_button)
                        time.sleep(3)

                        # Wait for table
                        table = wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
                        rows = table.find_elements(By.TAG_NAME, "tr")

                        for row in rows[1:]:  # Skip header
                            cols = [col.text.strip() for col in row.find_elements(By.TAG_NAME, "td")]
                            if len(cols) >= 3:
                                all_polling_units.append({
                                    "LGA": lga,
                                    "Ward": ward,
                                    "Polling Unit Code": cols[1] if len(cols) > 1 else "",
                                    "Polling Unit Name": cols[0]
                                })

                        print(f"     ✓ Ward '{ward}' → {len(rows)-1} units")

                    except Exception as e:
                        print(f"     ⚠️ Ward '{ward}' failed: {e}")
                        continue

            except Exception as e:
                print(f"  ⚠️ LGA '{lga}' failed: {e}")
                continue

        # --- Save all to Excel ---
        os.makedirs(OUTPUT_FOLDER, exist_ok=True)

        # Convert to DataFrames
        df1 = pd.DataFrame(lga_list).drop_duplicates()
        df2 = pd.DataFrame(ward_list).drop_duplicates()
        df3 = pd.DataFrame(all_polling_units)

        file_path = os.path.join(OUTPUT_FOLDER, f"{state_name}_polling_units.xlsx")
        with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
            df1.to_excel(writer, index=False, sheet_name="State_LGA")
            df2.to_excel(writer, index=False, sheet_name="State_LGA_Wards")
            df3.to_excel(writer, index=False, sheet_name="Polling_Units")

        print(f"💾 Saved '{file_path}' successfully with {len(df1)} LGAs, {len(df2)} wards, {len(df3)} polling units.")

    except Exception as e:
        print(f"❌ State scraping error: {e}")


# ==============================
# MAIN SCRAPING CONTROLLER
# ==============================
def main():
    driver = create_driver()
    try:
        driver.get(BASE_URL)
        time.sleep(5)

        all_states = [
            "ABIA", "ADAMAWA", "AKWA IBOM", "ANAMBRA", "BAUCHI", "BAYELSA", "BENUE", "BORNO",
            "CROSS RIVER", "DELTA", "EBONYI", "EDO", "EKITI", "ENUGU", "FEDERAL CAPITAL TERRITORY",
            "GOMBE", "IMO", "JIGAWA", "KADUNA", "KANO", "KATSINA", "KEBBI", "KOGI", "KWARA", "LAGOS",
            "NASARAWA", "NIGER", "OGUN", "ONDO", "OSUN", "OYO", "PLATEAU", "RIVERS", "SOKOTO",
            "TARABA", "YOBE", "ZAMFARA"
        ]

        print(f"🔹 Starting scraping for {len(all_states)} states...")

        for state in all_states:
            print(f"\n===============================")
            print(f"🔹 Scraping polling units for {state}...")
            print(f"===============================")
            try:
                scrape_state(driver, state)
            except Exception as e:
                print(f"❌ Error scraping {state}: {e}")
            time.sleep(5)

        print("\n✅ All states processed successfully.")
    finally:
        driver.quit()


# ==============================
# ENTRY POINT
# ==============================
if __name__ == "__main__":
    main()


🔹 Starting scraping for 37 states...

🔹 Scraping polling units for ABIA...
✅ Selected state: ABIA
Found 17 LGAs in ABIA
  ▪ LGA: ABA NORTH → 12 wards found
     ✓ Ward 'EZIAMA' → 52 units
     ✓ Ward 'INDUSTRIAL AREA' → 43 units
     ✓ Ward 'OSUSU I' → 45 units
     ✓ Ward 'OSUSU II' → 36 units
     ✓ Ward 'ST.EUGENES BY OKIGWE RD.' → 29 units
     ✓ Ward 'URATTA' → 41 units
     ✓ Ward 'OLD ABA GRA' → 37 units
     ✓ Ward 'UMUOLA' → 43 units
     ✓ Ward 'ARIARIA MARKET' → 91 units
     ✓ Ward 'OGBOR I' → 31 units
     ✓ Ward 'OGBOR II' → 26 units
     ✓ Ward 'UMUOGOR' → 24 units
  ▪ LGA: ABA SOUTH → 12 wards found
     ✓ Ward 'EZIUKWU' → 56 units
     ✓ Ward 'ASA' → 37 units
     ✓ Ward 'ENYIMBA' → 50 units
     ✓ Ward 'NGWA' → 67 units
     ✓ Ward 'OHAZU I' → 46 units
     ✓ Ward 'OHAZU II' → 30 units
     ✓ Ward 'IGWEBUIKE' → 29 units
     ✓ Ward 'EKEOHA' → 40 units
     ✓ Ward 'GLOUCESTER' → 36 units
     ✓ Ward 'MOSQUE' → 36 units
     ✓ Ward 'ABA RIVER' → 39 units
     ✓ Ward 'AB

In [ ]:
"""
Video Understanding and Prompt Generation Script
=================================================

This script:
1. Takes a single video file or a folder of videos.
2. Detects key scenes (shot boundaries).
3. Extracts representative keyframes from each scene.
4. Describes each keyframe using a pretrained image captioning model (BLIP).
5. Summarizes the video with a pretrained summarization model (BART).
6. Generates a creative prompt describing how to recreate or augment the video.

Dependencies:
--------------
pip install opencv-python pySceneDetect torch torchvision transformers pillow tqdm

Author: Jare (GPT-5 assisted)
"""

import os
import cv2
import torch
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration, pipeline
from  scenedetect import VideoManager, SceneManager
from scenedetect.detectors import ContentDetector


# ============================================================
# Configuration
# ============================================================
INPUT_PATH = "C:/Users/JARE WORKS/Videos/Captures"  # Path to a folder OR single video file
OUTPUT_DIR = "video_analysis_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 1. Scene Detection and Keyframe Extraction
# ============================================================
def extract_keyframes(video_path, output_dir):
    """
    Extracts one representative keyframe per detected scene.
    """
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    video_output_dir = os.path.join(output_dir, video_name)
    os.makedirs(video_output_dir, exist_ok=True)

    video_manager = VideoManager([video_path])
    scene_manager = SceneManager()
    scene_manager.add_detector(ContentDetector(threshold=30.0))

    video_manager.start()
    scene_manager.detect_scenes(frame_source=video_manager)
    scene_list = scene_manager.get_scene_list()

    cap = cv2.VideoCapture(video_path)
    keyframes = []

    for i, (start, end) in enumerate(scene_list):
        frame_number = (start.get_frames() + end.get_frames()) // 2
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = cap.read()
        if ret:
            filename = os.path.join(video_output_dir, f"scene_{i:03d}.jpg")
            cv2.imwrite(filename, frame)
            keyframes.append(filename)

    cap.release()
    return keyframes


# ============================================================
# 2. Frame Captioning using BLIP
# ============================================================
def describe_frames(frame_paths):
    """
    Generates captions for a list of keyframe images.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained(
        "Salesforce/blip-image-captioning-base"
    ).to(device)

    captions = []
    for frame in tqdm(frame_paths, desc="Describing keyframes"):
        image = cv2.cvtColor(cv2.imread(frame), cv2.COLOR_BGR2RGB)
        inputs = processor(image, return_tensors="pt").to(device)
        out = model.generate(**inputs, max_length=50)
        caption = processor.decode(out[0], skip_special_tokens=True)
        captions.append(caption)

    return captions


# ============================================================
# 3. Video-Level Summarization
# ============================================================
def summarize_video(captions):
    """
    Summarizes all captions into a coherent story using a BART summarizer.
    """
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
    combined_text = " ".join(captions)
    summary = summarizer(
        combined_text, max_length=200, min_length=60, do_sample=False
    )[0]["summary_text"]
    return summary


# ============================================================
# 4. Prompt Generation
# ============================================================
def generate_creation_prompt(summary):
    """
    Generates a prompt to recreate or augment the described video.
    """
    prompt = (
        "Recreate a video based on the following detailed description:\n\n"
        f"{summary}\n\n"
        "Ensure realistic lighting, natural motion, and emotional continuity. "
        "You may expand on this concept with creative camera angles or added context."
    )
    return prompt


# ============================================================
# 5. Full Video Analysis Pipeline
# ============================================================
def analyze_video(video_path, output_dir):
    """
    Runs the full pipeline on one video.
    """
    print(f"\n=== Processing Video: {video_path} ===")
    keyframes = extract_keyframes(video_path, output_dir)

    if not keyframes:
        print("No scenes detected, skipping.")
        return

    captions = describe_frames(keyframes)
    summary = summarize_video(captions)
    prompt = generate_creation_prompt(summary)

    # Save outputs
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    text_output = os.path.join(output_dir, f"{video_name}_summary.txt")
    with open(text_output, "w", encoding="utf-8") as f:
        f.write("=== Video Summary ===\n")
        f.write(summary + "\n\n")
        f.write("=== Recreation Prompt ===\n")
        f.write(prompt)

    print(f"\n✅ Analysis completed for {video_name}")
    print(f"Summary and prompt saved to: {text_output}")
    print("\n--- SUMMARY ---")
    print(summary)
    print("\n--- CREATION PROMPT ---")
    print(prompt)


# ============================================================
# 6. Entry Point
# ============================================================
if __name__ == "__main__":
    if os.path.isfile(INPUT_PATH):
        analyze_video(INPUT_PATH, OUTPUT_DIR)
    elif os.path.isdir(INPUT_PATH):
        video_files = [
            os.path.join(INPUT_PATH, f)
            for f in os.listdir(INPUT_PATH)
            if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
        ]
        for video in video_files:
            analyze_video(video, OUTPUT_DIR)
    else:
        print("❌ Invalid input path. Please provide a video file or a folder of videos.")
